In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType

from dataclasses import dataclass
from typing import Optional, Dict, Any, List
import re
from sklearn.metrics import mean_absolute_error, cohen_kappa_score

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,   # thêm dòng này
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # khuyên dùng bản này cho mốc 3B
TRAIN_CSV = "/content/ielts_train_df.csv"
VAL_CSV   = "/content/ielts_val_df.csv"
TEST_CSV  = "/content/ielts_test_df.csv"

MAX_LENGTH = 1024
SEED = 42

BATCH_SIZE = 1
GRAD_ACCUM = 8

LR = 5e-5
EPOCHS = 5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./qwen25_3b_ielts_4criteria"

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

set_seed(SEED)

GRA_FEATURE_COLS = [
    "gf_word_count",
    "gf_sentence_count",
    "gf_avg_sentence_len",
    "gf_short_sentence_ratio",
    "gf_long_sentence_ratio",
    "gf_punct_density",
    "gf_repeated_punct_ratio",
    "gf_lowercase_sent_start_ratio",
    "gf_lowercase_i_ratio",
    "gf_repeated_word_ratio",
    "gf_missing_terminal_punct",
]

CRITERION_WEIGHTS = [1.0, 1.0, 1.0, 2.0]
HEAD_DROPOUT = 0.1
BIAS_LOSS_WEIGHT = 0.02

In [ ]:
def extract_grammar_features(text: str):
    text = str(text).strip()

    # words
    words = re.findall(r"\b[\w']+\b", text)
    word_count = max(len(words), 1)

    # sentences
    sentence_candidates = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentence_candidates if s.strip()]
    sentence_count = max(len(sentences), 1)

    # avg sentence length
    avg_sentence_len = word_count / sentence_count

    # short / long sentence ratio
    sent_word_counts = []
    for s in sentences:
        sw = re.findall(r"\b[\w']+\b", s)
        sent_word_counts.append(len(sw))

    if len(sent_word_counts) == 0:
        sent_word_counts = [word_count]

    short_sentence_ratio = np.mean([c < 8 for c in sent_word_counts])
    long_sentence_ratio = np.mean([c > 25 for c in sent_word_counts])

    # punctuation density
    punct_count = len(re.findall(r"[,:;()\-\"]", text))
    punct_density = punct_count / word_count

    # repeated punctuation like !! ?? ... ,, ;;
    repeated_punct_count = len(re.findall(r"([!?.,;:])\1+", text))
    repeated_punct_ratio = repeated_punct_count / sentence_count

    # lowercase sentence start ratio
    lowercase_sent_starts = 0
    for s in sentences:
        s = s.strip()
        if len(s) > 0 and s[0].islower():
            lowercase_sent_starts += 1
    lowercase_sent_start_ratio = lowercase_sent_starts / sentence_count

    # lowercase standalone "i"
    lowercase_i_count = len(re.findall(r"\bi\b", text))
    lowercase_i_ratio = lowercase_i_count / word_count

    # repeated adjacent words: "the the", "is is"
    repeated_word_count = 0
    lower_words = [w.lower() for w in words]
    for i in range(1, len(lower_words)):
        if lower_words[i] == lower_words[i - 1]:
            repeated_word_count += 1
    repeated_word_ratio = repeated_word_count / word_count

    # missing terminal punctuation
    missing_terminal_punct = 0.0 if re.search(r"[.!?]\s*$", text) else 1.0

    feats = [
        float(word_count),
        float(sentence_count),
        float(avg_sentence_len),
        float(short_sentence_ratio),
        float(long_sentence_ratio),
        float(punct_density),
        float(repeated_punct_ratio),
        float(lowercase_sent_start_ratio),
        float(lowercase_i_ratio),
        float(repeated_word_ratio),
        float(missing_terminal_punct),
    ]
    return feats

In [ ]:
# 1. Load dữ liệu gốc
train_df = pd.read_csv(TRAIN_CSV, engine="python", on_bad_lines="skip")
val_df = pd.read_csv(VAL_CSV, engine="python", on_bad_lines="skip") if os.path.exists(VAL_CSV) else None
test_df = pd.read_csv(TEST_CSV, engine="python", on_bad_lines="skip") if os.path.exists(TEST_CSV) else None

# 2. Tách tập validation nếu chưa có
if val_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

needed_cols = ["prompt", "essay"] + TARGET_COLS

def robust_clean_df(df):
    if df is None:
        return None

    df = df[needed_cols].copy()

    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["essay"] = df["essay"].astype(str).str.strip()

    df = df[(df["prompt"].str.len() > 0) & (df["essay"].str.len() > 20)].reset_index(drop=True)

    for col in TARGET_COLS:
        df[col] = df[col].clip(0.0, 9.0)

    return df

train_df = robust_clean_df(train_df)
val_df = robust_clean_df(val_df)
test_df = robust_clean_df(test_df)

print(f"Train shape: {train_df.shape if train_df is not None else 0}")
print(f"Val shape: {val_df.shape if val_df is not None else 0}")
print(train_df[TARGET_COLS].head())

Train shape: (6618, 6)
Val shape: (827, 6)
    TR   CC   LR  GRA
0  6.0  6.0  6.0  6.0
1  6.5  6.5  6.5  6.5
2  7.0  7.0  7.0  7.0
3  5.0  5.0  5.0  5.0
4  4.5  4.5  4.5  4.5


In [ ]:
def build_input_text(row):
    prompt = str(row["prompt"]).strip()
    essay = str(row["essay"]).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

train_df["text"] = train_df.apply(build_input_text, axis=1)
val_df["text"] = val_df.apply(build_input_text, axis=1)

if test_df is not None:
    test_df["text"] = test_df.apply(build_input_text, axis=1)

# ===== Grammar features =====
train_gra_feats = np.array([extract_grammar_features(x) for x in train_df["essay"]], dtype=np.float32)
val_gra_feats = np.array([extract_grammar_features(x) for x in val_df["essay"]], dtype=np.float32)

if test_df is not None:
    test_gra_feats = np.array([extract_grammar_features(x) for x in test_df["essay"]], dtype=np.float32)

# Standardize bằng train stats
gra_feat_mean = train_gra_feats.mean(axis=0)
gra_feat_std = train_gra_feats.std(axis=0)
gra_feat_std = np.where(gra_feat_std < 1e-6, 1.0, gra_feat_std)

train_gra_feats = (train_gra_feats - gra_feat_mean) / gra_feat_std
val_gra_feats = (val_gra_feats - gra_feat_mean) / gra_feat_std

if test_df is not None:
    test_gra_feats = (test_gra_feats - gra_feat_mean) / gra_feat_std

train_df["gra_features"] = train_gra_feats.tolist()
val_df["gra_features"] = val_gra_feats.tolist()

if test_df is not None:
    test_df["gra_features"] = test_gra_feats.tolist()

def make_labels(df):
    df = df.copy()
    df["labels"] = (df[TARGET_COLS] / 9.0).values.tolist()
    return df

train_df = make_labels(train_df)
val_df = make_labels(val_df)
if test_df is not None:
    test_df = make_labels(test_df)

print("Num grammar features:", len(GRA_FEATURE_COLS))
print("Example gra_features:", train_df["gra_features"].iloc[0])

In [ ]:
train_ds = Dataset.from_pandas(train_df[["text", "labels", "gra_features"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels", "gra_features"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

if test_df is not None:
    test_ds = Dataset.from_pandas(test_df[["text", "labels", "gra_features"]], preserve_index=False)
    dataset_dict["test"] = test_ds

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format(type="torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/6618 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

In [ ]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        self.backbone = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        self.backbone.config.use_cache = False

        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        )
        self.backbone = get_peft_model(self.backbone, lora_config)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        # 3 nhánh regression thường
        self.head_tr = nn.Linear(hidden_size, 1)
        self.head_cc = nn.Linear(hidden_size, 1)
        self.head_lr = nn.Linear(hidden_size, 1)

        # nhánh GRA regression + grammar features
        self.gra_feat_dim = len(GRA_FEATURE_COLS)
        self.gra_mlp = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )

        self.reg_activation = nn.Sigmoid()

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def forward(self, input_ids=None, attention_mask=None, gra_features=None, labels=None, **kwargs):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        # 3 tiêu chí đầu: regression 0..1
        tr = self.reg_activation(self.head_tr(pooled))
        cc = self.reg_activation(self.head_cc(pooled))
        lr = self.reg_activation(self.head_lr(pooled))

        # GRA: regression 0..1
        if gra_features is None:
            raise ValueError("gra_features is required")

        gra_features = gra_features.to(device=pooled.device, dtype=pooled.dtype)
        gra_input = torch.cat([pooled, gra_features], dim=1)
        gra = self.reg_activation(self.gra_mlp(gra_input))

        combined_logits = torch.cat([tr, cc, lr, gra], dim=1)

        return {
            "logits": combined_logits,      # shape [B, 4]
            "reg_logits": combined_logits,  # giữ key này để trainer không vỡ
        }


model = QwenForIELTSMultiTask(MODEL_NAME, tokenizer, head_dropout=HEAD_DROPOUT)
model.backbone.print_trainable_parameters()
print("Heads are trainable by default.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,694,592 || all params: 3,089,641,472 || trainable%: 0.1196


In [ ]:
import torch
import torch.nn as nn
from transformers import Trainer

class IELTSMultiTaskTrainer(Trainer):
    def __init__(
        self,
        *args,
        criterion_weights=None,
        bias_loss_weight=0.02,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        if criterion_weights is None:
            criterion_weights = [1.0, 1.0, 1.0, 2.0]

        self.criterion_weights = torch.tensor(criterion_weights, dtype=torch.float32)
        self.bias_loss_weight = bias_loss_weight

        # regression cho cả 4 tiêu chí
        self.reg_loss_fn = nn.HuberLoss(delta=0.08, reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs["logits"]   # [B, 4]
        labels = labels.to(logits.dtype)

        # loss từng tiêu chí
        per_dim_loss = self.reg_loss_fn(logits, labels)  # [B, 4]

        weights = self.criterion_weights.to(
            device=logits.device,
            dtype=logits.dtype
        ).view(1, -1)  # [1, 4]

        # weighted mean loss
        weighted_loss = (per_dim_loss * weights).sum() / (
            weights.sum().clamp_min(1e-8) * logits.size(0)
        )

        # bias penalty: tránh lệch mean prediction
        pred_mean = logits.mean(dim=0)
        label_mean = labels.mean(dim=0)
        bias_loss = torch.abs(pred_mean - label_mean).mean()

        loss = weighted_loss + self.bias_loss_weight * bias_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
@dataclass
class RegressionCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.stack([
            f["labels"] if isinstance(f["labels"], torch.Tensor)
            else torch.tensor(f["labels"], dtype=torch.float)
            for f in features
        ]).float()

        gra_features = torch.stack([
            f["gra_features"] if isinstance(f["gra_features"], torch.Tensor)
            else torch.tensor(f["gra_features"], dtype=torch.float)
            for f in features
        ]).float()

        batch = self.tokenizer.pad(
            [{k: v for k, v in f.items() if k not in ["labels", "gra_features"]} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        batch["gra_features"] = gra_features
        return batch

data_collator = RegressionCollator(tokenizer)

In [11]:
def round_to_half(x):
    return np.round(x * 2) / 2

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.float32)

    preds = preds * 9.0
    labels = labels * 9.0

    preds = np.clip(preds, 0.0, 9.0)
    labels = np.clip(labels, 0.0, 9.0)

    preds_half = round_to_half(preds)

    maes = []
    qwks = []

    for i in range(labels.shape[1]):
        maes.append(mean_absolute_error(labels[:, i], preds_half[:, i]))

        y_true = np.round(labels[:, i] * 2).astype(int)
        y_pred = np.round(preds_half[:, i] * 2).astype(int)

        qwks.append(cohen_kappa_score(y_true, y_pred, weights="quadratic"))

    within_05 = np.mean(np.abs(preds_half - labels) <= 0.5)

    return {
        "mean_mae": float(np.mean(maes)),
        "mean_qwk": float(np.mean(qwks)),
        "tr_qwk": float(qwks[0]),
        "cc_qwk": float(qwks[1]),
        "lr_qwk": float(qwks[2]),
        "gra_qwk": float(qwks[3]),
        "within_0.5_acc": float(within_05),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="mean_qwk",
    greater_is_better=True,

    bf16=True,
    fp16=False,

    gradient_checkpointing=False,
    remove_unused_columns=False,
    report_to="none",
    save_total_limit=2,
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = max(
    1, math.ceil(len(tokenized_ds["train"]) / (BATCH_SIZE * GRAD_ACCUM))
)

num_training_steps = num_update_steps_per_epoch * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps
)

In [ ]:
trainer = IELTSMultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    criterion_weights=CRITERION_WEIGHTS,
    bias_loss_weight=BIAS_LOSS_WEIGHT,
    optimizers=(optimizer, scheduler),
)

In [15]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(debug_loader))
print(batch.keys())
print(batch["labels"])
print(batch["labels"].shape)
print(batch["labels"].dtype)

KeysView({'input_ids': tensor([[ 42347,   3361,   2828,    921,   8441,   5837,    525,  10164,    264,
           6765,   3311,    315,   3220,    389,  12613,    862,  27550,    311,
           1896,    949,    304,   1045,  15245,   9833,  42582,     13,  25028,
          17585,    429,    432,   1035,    387,   2664,    421,   1493,   5837,
            646,   8329,    279,   3220,    389,   2841,    311,   1896,    949,
            304,   9833,     13,   2014,   1128,  12818,    653,    498,   7503,
            476,  28295,   1939,     58,   9996,   3022,    921,   3862,    525,
           2696,    315,   5837,    304,    279,   1879,    879,    525,   1667,
           2244,   3311,    315,   3220,    389,  42649,    315,    862,   7263,
             13,    358,   7503,    429,    807,   1265,   1191,  10164,   3220,
            389,   2841,  10552,   7488,    304,    429,   1616,    807,    686,
           1191,    279,  50375,    315,   4124,  17628,    304,    862,   2272,
     

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [18]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:", val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
    print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.007700358517467976, 'eval_mean_mae': 0.8146916478872299, 'eval_mean_qwk': 0.5512190672513295, 'eval_tr_qwk': 0.5655727505061547, 'eval_cc_qwk': 0.5525419438303159, 'eval_lr_qwk': 0.5506673233471044, 'eval_gra_qwk': 0.5360942513217433, 'eval_within_0.5_acc': 0.5389963724304716, 'eval_runtime': 66.563, 'eval_samples_per_second': 12.424, 'eval_steps_per_second': 12.424, 'epoch': 5.0}


early stopping required metric_for_best_model, but did not find eval_mean_qwk so early stopping is disabled


Test metrics: {'test_loss': 0.007885178551077843, 'test_mean_mae': 0.8140096515417099, 'test_mean_qwk': 0.5585495155902782, 'test_tr_qwk': 0.577194502909496, 'test_cc_qwk': 0.5613661237914654, 'test_lr_qwk': 0.5771517919429552, 'test_gra_qwk': 0.5184856437171965, 'test_within_0.5_acc': 0.5428743961352657, 'test_runtime': 67.4399, 'test_samples_per_second': 12.278, 'test_steps_per_second': 12.278, 'epoch': 5.0}


In [17]:
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")

('./qwen25_3b_ielts_4criteria/best_model/tokenizer_config.json',
 './qwen25_3b_ielts_4criteria/best_model/chat_template.jinja',
 './qwen25_3b_ielts_4criteria/best_model/tokenizer.json')

In [19]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

OUTPUT_DIR = "./qwen25_3b_ielts_4criteria"

shutil.make_archive("/content/model_output", 'zip', OUTPUT_DIR)

shutil.copy("/content/model_output.zip",
            "/content/drive/MyDrive/model_output.zip")

print("Model zipped and saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model zipped and saved to Google Drive!


In [16]:
import os
import gc
import torch

# Nếu muốn train tiếp thêm, tăng tổng số epoch lên
# Ví dụ trước đó đã train 3 epoch, giờ muốn train tới epoch 6:
trainer.args.num_train_epochs = 10

# Tự tìm checkpoint mới nhất trong OUTPUT_DIR
checkpoint_dirs = [
    os.path.join(OUTPUT_DIR, d)
    for d in os.listdir(OUTPUT_DIR)
    if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
]

if not checkpoint_dirs:
    raise ValueError(f"Không tìm thấy checkpoint nào trong {OUTPUT_DIR}")

latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
print("Resuming from:", latest_checkpoint)

gc.collect()
torch.cuda.empty_cache()

train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

print("Resume train xong.")
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming from: ./qwen25_3b_ielts_4criteria/checkpoint-4140


Epoch,Training Loss,Validation Loss,Mean Mae,Mean Qwk,Tr Qwk,Cc Qwk,Lr Qwk,Gra Qwk,Within 0.5 Acc
6,0.057244,0.007818,0.813785,0.557399,0.574845,0.558936,0.565694,0.530121,0.548972
7,0.048815,0.008436,0.861094,0.529710,0.547630,0.491138,0.537700,0.542371,0.505744


Epoch,Training Loss,Validation Loss,Mean Mae,Mean Qwk,Tr Qwk,Cc Qwk,Lr Qwk,Gra Qwk,Within 0.5 Acc
6,0.057244,0.007818,0.813785,0.557399,0.574845,0.558936,0.565694,0.530121,0.548972
7,0.048815,0.008436,0.861094,0.529710,0.547630,0.491138,0.537700,0.542371,0.505744
8,0.065996,0.008051,0.828446,0.534971,0.556587,0.525846,0.532240,0.525212,0.536880


Resume train xong.
TrainOutput(global_step=6624, training_loss=0.02185134117487044, metrics={'train_runtime': 8926.312, 'train_samples_per_second': 7.414, 'train_steps_per_second': 0.928, 'total_flos': 3.412890316575867e+17, 'train_loss': 0.02185134117487044, 'epoch': 8.0})
